# Ballistic backmapping quickstart (general spacecraft)

This notebook demonstrates how to run the repository's general ballistic backmapping pipeline and inspect outputs.

It calls: `functions/sc_pos/ballistic_backmap.py`


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Adjust these values as needed for your dataset/spacecraft
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'Notebooks_Examples' else Path.cwd().resolve()
final_pkl = repo_root / 'examples' / 'SOLO' / '2022-10-01_00-00-00_2022-10-03_00-00-00_sc_0' / 'final.pkl'
outdir = repo_root / 'examples' / 'figures' / 'ballistic_backmap_demo'
target = 'SOLO'  # e.g. SOLO, PSP, WIND, ACE, -144
target_label = 'Solar Orbiter'
start = '2022-10-01T00:00:00Z'
stop = '2022-10-02T00:00:00Z'
cadence = '60min'
smooth = '12h'

print('repo_root:', repo_root)
print('final_pkl exists:', final_pkl.exists())
print('outdir:', outdir)


In [ ]:
cmd = [
    sys.executable,
    str(repo_root / 'functions' / 'sc_pos' / 'ballistic_backmap.py'),
    '--final_pkl', str(final_pkl),
    '--outdir', str(outdir),
    '--target', target,
    '--target_label', target_label,
    '--start', start,
    '--stop', stop,
    '--cadence', cadence,
    '--smooth', smooth,
    '--cache_ephem',
]

print('Running command:
', ' '.join(cmd))
res = subprocess.run(cmd, cwd=repo_root, text=True, capture_output=True)
print('Return code:', res.returncode)
if res.stdout:
    print('--- stdout ---')
    print(res.stdout)
if res.stderr:
    print('--- stderr ---')
    print(res.stderr)

if res.returncode != 0:
    raise RuntimeError('Backmapping command failed. See stdout/stderr above.')


In [ ]:
pngs = [
    outdir / 'source_surface_polarity.png',
    outdir / 'source_surface_speed.png',
    outdir / 'source_surface_ram_pressure.png',
]

fig, axes = plt.subplots(3, 1, figsize=(12, 12), constrained_layout=True)
for ax, p in zip(axes, pngs):
    if not p.exists():
        ax.set_title(f'Missing output: {p.name}')
        ax.axis('off')
        continue
    img = plt.imread(p)
    ax.imshow(img)
    ax.set_title(p.name)
    ax.axis('off')
plt.show()


In [ ]:
ts_path = outdir / 'ballistic_backmap_timeseries.pkl'
ts = pd.read_pickle(ts_path)
print('timeseries shape:', ts.shape)
display(ts[[c for c in ['Br_large', 'Vr_large', 'Np_large', 'phi_src', 'lat_src', 'Pram'] if c in ts.columns]].head())
